# 01 - Senior et al. 2020: AlphaFold CASP13 Reproduction

This notebook is a **from-scratch PyTorch reproduction** of the first AlphaFold paper, not a wrapper around DeepMind's CASP13 code or weights.

The Senior et al. system can be decomposed into four reproducible ideas:

1. Build leakage-controlled sequence/template features for CASP13 targets.
2. Train a residual network that predicts inter-residue distance distributions, torsions, and auxiliary structure features.
3. Convert predicted distance probabilities into a differentiable potential of mean force.
4. Optimize 3D coordinates under that potential and score them against CASP13 structures.

The first faithful target is "same benchmark, same information boundary." The improvement target is "beat their benchmark with clearly labeled enhanced experiments."

## Data and model layout

All input data and trained checkpoints are ours:

- `data/senior_2020/raw`: CASP13 FASTA, ground-truth structures, dated PDB/CATH snapshots, and MSA database snapshots.
- `data/senior_2020/features`: tensorized MSA/template/contact labels.
- `models/senior_2020/checkpoints`: our PyTorch checkpoints.
- `results/senior_2020`: distograms, optimized PDBs, and score tables.

No official AlphaFold API, Docker runner, model parameters, or inference script is used.

## Extended Data Fig. 1 Reproduction Checklist

This checklist maps the notebook against the Senior et al. AlphaFold method schematic in Extended Data Fig. 1. It is deliberately strict: a checked item means the notebook has an executable implementation of that step in our own PyTorch/data code; a partial item means the notebook has the interface or simplified analogue but not the faithful paper-level implementation yet.

| Extended Data Fig. 1 component | Status | Notebook coverage | What remains for faithful reproduction |
|---|---:|---|---|
| CASP-era information boundary | Implemented | `benchmark_contract.json` records CASP13 target family, template cutoff, CATH cutoff, and database snapshot labels. | Replace labels with verified downloaded database manifests and checksums. |
| Sequence database search / MSA construction | Missing | The feature directory is defined, and synthetic tensors keep the notebook runnable. | Add HHblits/JackHMMER/PSI-BLAST runners, cluster-safe job scripts, database paths, e-value/coverage filters, and cached `.a3m`/`.sto` outputs. |
| MSA-derived 1D and 2D features | Partial | The dataset expects `msa_profile` and `msa_covariance`-like pair tensors. | Implement the exact Senior-style sequence profiles, covariance/coupling summaries, sequence separation features, masking, and normalization from real alignments. |
| Template search and template pair features | Partial | `template_pair` is part of the tensor contract. | Add dated template search, hit filtering by cutoff, alignment mapping, template distance/orientation features, and leakage auditing. |
| Residual 2D convolutional network | Partial | The notebook implements a PyTorch residual 2D CNN over `[L, L, C]` pair tensors. | Match the Extended Data Fig. 1 block more closely: dimension-reduction layers, dilated convolution schedule, bypass/residual layout, depth/width, activations, normalization, ensembling, and initialization. |
| Distogram prediction | Partial | The network predicts symmetric categorical distance logits and trains with pairwise cross-entropy. | Use paper-faithful atom choice, distance range, bin count, masks, label generation, class weighting, and calibration checks. |
| Torsion-angle / local-geometry heads | Missing | No torsion or local backbone heads are currently trained. | Add torsion labels, angular losses, local structure terms, and their contribution to final coordinate generation. |
| Potential of mean force from predictions | Partial | Predicted log-probabilities become a differentiable coordinate energy with a reference correction. | Implement the paper-level reference distogram, potential interpolation/smoothing, weighting, long-range emphasis, and all auxiliary energy terms. |
| Structure realization / optimization | Partial | Coordinates are optimized directly with PyTorch gradients and chain-length regularization. | Reproduce the original realization procedure more faithfully, including torsion-space/backbone constraints, multiple starts, annealing or staged optimization, stereochemical terms, and final relaxation. |
| CASP13 evaluation | Partial | The notebook has a metric registry and example-protein comparison plots. | Add the exact CASP13 free-modelling domain list, official target/domain parsing, GDT_TS/TM-score/lDDT executables where available, and paper-style aggregate tables. |
| Comparison to Senior et al. predictions | Partial | The manifest can compare our PDBs with reference structures and optional Senior prediction PDBs. | Materialize official/paper prediction files for all benchmark targets and compute paired deltas across the full benchmark. |

Current interpretation: this notebook is a runnable AF1-like reproduction scaffold, not yet a complete reproduction of every box in Extended Data Fig. 1. The next milestone is to turn the missing data pipeline rows into real cluster jobs, because all downstream architectural and scoring claims depend on having faithful MSA, template, and label tensors.

In [1]:
from __future__ import annotations

import json
import math
import os
import random
import shlex
import subprocess
import time
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

PROJECT_ROOT = Path.cwd()
DATA_ROOT = PROJECT_ROOT / "data"
MODEL_ROOT = PROJECT_ROOT / "models"
RESULTS_ROOT = PROJECT_ROOT / "results"
RUNS_ROOT = PROJECT_ROOT / "runs"

for path in [DATA_ROOT, MODEL_ROOT, RESULTS_ROOT, RUNS_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

def latest_environment_report() -> dict:
    report_dir = DATA_ROOT / "environment_reports"
    reports = sorted(report_dir.glob("environment_report_*_utc.json"))
    if not reports:
        return {}
    return json.loads(reports[-1].read_text(encoding="utf-8"))

ENV_REPORT = latest_environment_report()

def seed_everything(seed: int = 7) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def device() -> torch.device:
    return torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

def write_text(path: Path, text: str) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")
    return path

def run(cmd: list[str], *, cwd: Path | None = None, timeout: int = 30, dry_run: bool = True):
    print("$", " ".join(shlex.quote(str(x)) for x in cmd))
    if dry_run:
        print("DRY_RUN=True, command was not executed.")
        return None
    completed = subprocess.run(cmd, cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=timeout, check=False)
    print(completed.stdout)
    if completed.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {completed.returncode}")
    return completed

def gpu_summary() -> str:
    devices = ENV_REPORT.get("torch", {}).get("devices", [])
    if devices:
        first = devices[0]
        return f"{first.get('name')} / {first.get('total_memory_gb')} GB / cc {first.get('major')}.{first.get('minor')}"
    return "No saved GPU report found."

seed_everything(7)
print(f"Project root: {PROJECT_ROOT}")
print(f"Device: {device()}")
print(f"Saved cluster GPU: {gpu_summary()}")

Project root: /workspace
Device: cuda:0
Saved cluster GPU: NVIDIA A100-SXM4-80GB / 79.251 GB / cc 8.0


In [2]:
PAPER = "senior_2020"
DATA_DIR = DATA_ROOT / PAPER
MODEL_DIR = MODEL_ROOT / PAPER
RESULT_DIR = RESULTS_ROOT / PAPER
RUN_DIR = RUNS_ROOT / PAPER

paths = {
    "raw": DATA_DIR / "raw",
    "features": DATA_DIR / "features",
    "labels": DATA_DIR / "labels",
    "checkpoints": MODEL_DIR / "checkpoints",
    "distograms": RESULT_DIR / "distograms",
    "structures": RESULT_DIR / "structures",
    "metrics": RESULT_DIR / "metrics",
    "slurm": RUN_DIR / "slurm",
}
for p in paths.values():
    p.mkdir(parents=True, exist_ok=True)
print(json.dumps({k: str(v) for k, v in paths.items()}, indent=2))

{
  "raw": "/workspace/data/senior_2020/raw",
  "features": "/workspace/data/senior_2020/features",
  "labels": "/workspace/data/senior_2020/labels",
  "checkpoints": "/workspace/models/senior_2020/checkpoints",
  "distograms": "/workspace/results/senior_2020/distograms",
  "structures": "/workspace/results/senior_2020/structures",
  "metrics": "/workspace/results/senior_2020/metrics",
  "slurm": "/workspace/runs/senior_2020/slurm"
}


## Step 1 - Benchmark contract

The contract below is the guardrail against accidental leakage. A faithful Senior reproduction should not train on structures that post-date CASP13 and should not use sequence/template databases newer than the paper-era versions. Later, enhanced experiments may use modern databases, but those rows must be marked `faithful=False`.

Scientifically, this step defines the counterfactual question: "What could the method have known at CASP13 time?" Protein-structure predictors can silently improve just by seeing homologous future structures or larger modern sequence databases, so the benchmark boundary is part of the experiment, not bookkeeping.

Computationally, we serialize the boundary as JSON so every later feature file, checkpoint, and score table can point back to the same data contract. Mathematically, it fixes the training and evaluation distributions: training samples are drawn from structures and alignments available before the cutoff, while CASP13 targets remain held out. Without that separation, a high TM-score would not estimate generalization.

In [3]:
contract = {
    "paper": "Senior et al. 2020",
    "benchmark": "CASP13 free modelling domains",
    "template_cutoff": "2018-03-15",
    "cath_cutoff": "2018-03-16",
    "uniclust30_version": "2017-10",
    "nr_snapshot": "2017-12-15",
    "implementation": "own_pytorch",
    "uses_official_weights_or_api": False,
}
write_text(DATA_DIR / "benchmark_contract.json", json.dumps(contract, indent=2))
print(json.dumps(contract, indent=2))

{
  "paper": "Senior et al. 2020",
  "benchmark": "CASP13 free modelling domains",
  "template_cutoff": "2018-03-15",
  "cath_cutoff": "2018-03-16",
  "uniclust30_version": "2017-10",
  "nr_snapshot": "2017-12-15",
  "implementation": "own_pytorch",
  "uses_official_weights_or_api": false
}


## Step 2 - Feature tensors

Senior-style distance prediction is naturally pairwise. We represent each protein as:

- `msa_profile`: `[L, A]`, amino-acid frequencies from the MSA.
- `msa_covariance`: `[L, L, C]`, cheap co-evolution/covariance summaries.
- `template_pair`: `[L, L, T]`, template-derived distances/masks if allowed by cutoff.
- `dist_bin`: `[L, L]`, supervised C-beta/C-alpha distance bin label.

The dataset class below is intentionally file-format simple. Feature builders can write `.npz` files from HHblits/JackHMMER/template tools later, while the model and training code stay stable.

Scientifically, the MSA summarizes evolutionary pressure: residues that mutate together often contact or constrain each other in 3D. Templates encode the separate hypothesis that homologous folds provide partial geometry when they are legitimately available before the cutoff.

Computationally, we turn variable biological evidence into dense tensors. The central tensor is `[L, L, C]`, where each entry describes a residue pair. Mathematically, the supervised label is a binned distance random variable `Y_ij`; the model will learn `p(Y_ij = b | MSA, templates)` for every residue pair `(i, j)`.

In [4]:
class SeniorFeatureDataset(Dataset):
    def __init__(self, feature_dir: Path, synthetic_if_empty: bool = True, n_synthetic: int = 8, length: int = 96, bins: int = 32):
        self.files = sorted(feature_dir.glob("*.npz"))
        self.synthetic_if_empty = synthetic_if_empty
        self.n_synthetic = n_synthetic
        self.length = length
        self.bins = bins

    def __len__(self):
        return len(self.files) if self.files else (self.n_synthetic if self.synthetic_if_empty else 0)

    def __getitem__(self, idx):
        if self.files:
            arr = np.load(self.files[idx])
            return {k: torch.as_tensor(arr[k]) for k in arr.files}

        L, B = self.length, self.bins
        msa_profile = F.one_hot(torch.randint(0, 21, (L,)), 21).float()
        msa_covariance = torch.randn(L, L, 16) * 0.1
        template_pair = torch.zeros(L, L, 8)
        pair = torch.cat([
            msa_profile[:, None, :].expand(L, L, 21),
            msa_profile[None, :, :].expand(L, L, 21),
            msa_covariance,
            template_pair,
        ], dim=-1)
        dist_bin = torch.randint(0, B, (L, L))
        dist_bin = torch.triu(dist_bin, diagonal=1) + torch.triu(dist_bin, diagonal=1).T
        return {"pair": pair, "dist_bin": dist_bin}

dataset = SeniorFeatureDataset(paths["features"])
batch = dataset[0]
print({k: tuple(v.shape) for k, v in batch.items()})

{'pair': (96, 96, 66), 'dist_bin': (96, 96)}


## Step 3 - Our Senior-style distogram network

The original system used deep residual networks over pair features. We keep that inductive bias: 2D convolutions operate on the residue-pair matrix, preserving local patterns in sequence separation and pairwise geometry. The network predicts distance bins for every residue pair.

Scientifically, a distogram is richer than a contact map because it preserves approximate geometry instead of only saying "near" or "far." That matters because many folds can satisfy the same binary contacts, but far fewer satisfy a full collection of distance distributions.

Computationally, the model is a 2D residual CNN over the residue-pair image. Residual connections let us stack many transformations without destroying gradient flow. Mathematically, the final head produces logits `z_ijb`, and `softmax_b(z_ijb)` is our estimate of the distance-bin distribution for pair `(i, j)`.

In [5]:
class Residual2DBlock(nn.Module):
    def __init__(self, channels: int, dilation: int = 1):
        super().__init__()
        pad = dilation
        self.net = nn.Sequential(
            nn.GroupNorm(8, channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=pad, dilation=dilation),
            nn.GroupNorm(8, channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1),
        )

    def forward(self, x):
        return x + self.net(x)


class SeniorDistogramNet(nn.Module):
    def __init__(self, pair_dim: int = 66, hidden: int = 128, blocks: int = 16, bins: int = 32):
        super().__init__()
        self.stem = nn.Conv2d(pair_dim, hidden, 1)
        self.blocks = nn.ModuleList([Residual2DBlock(hidden, dilation=1 + (i % 4)) for i in range(blocks)])
        self.dist_head = nn.Conv2d(hidden, bins, 1)

    def forward(self, pair):
        # pair: [B, L, L, C]
        x = pair.permute(0, 3, 1, 2).contiguous()
        x = self.stem(x)
        for block in self.blocks:
            x = block(x)
        logits = self.dist_head(x).permute(0, 2, 3, 1).contiguous()
        logits = 0.5 * (logits + logits.transpose(1, 2))
        return {"distogram_logits": logits}

model = SeniorDistogramNet().to(device())
with torch.no_grad():
    out = model(batch["pair"][None].to(device()))
print(tuple(out["distogram_logits"].shape))

(1, 96, 96, 32)


## Step 4 - Training objective

The distogram target is a categorical distance distribution. We train with cross-entropy over residue pairs, excluding the diagonal and optionally downweighting pairs close in sequence if we want to emphasize long-range structure. This is the learning step that turns MSA/template evidence into geometric constraints.

Scientifically, the network is asked to learn statistical regularities between evolutionary features and physical distances: co-evolving residues, conserved motifs, and template hints should increase probability mass on compatible distance bins.

Computationally, each protein contributes roughly `O(L^2)` residue-pair examples, so masking and batching matter on GPU. Mathematically, the loss is the negative log-likelihood `-sum_ij log p_theta(y_ij | x)` over valid residue pairs. Minimizing this trains the model to assign calibrated probability mass to the observed structure-derived bins.

In [6]:
def senior_loss(outputs, target_bins):
    logits = outputs["distogram_logits"]
    B, L, _, bins = logits.shape
    mask = torch.triu(torch.ones(L, L, device=logits.device, dtype=torch.bool), diagonal=1)
    loss = F.cross_entropy(logits[:, mask, :].reshape(-1, bins), target_bins[:, mask].reshape(-1))
    return loss

loader = DataLoader(dataset, batch_size=1, shuffle=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)
model.train()
for step, item in enumerate(loader):
    item = {k: v.to(device()) for k, v in item.items()}
    optimizer.zero_grad(set_to_none=True)
    loss = senior_loss(model(item["pair"]), item["dist_bin"].long())
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    print(f"step={step} loss={float(loss.detach().cpu()):.4f}")
    if step >= 2:
        break

step=0 loss=3.8327
step=1 loss=5.1782
step=2 loss=4.8997


## Step 5 - Potential of mean force and coordinate optimization

The network predicts probabilities, not coordinates. We reproduce the paper's core move: transform log-probabilities into a coordinate potential and optimize a chain. This code is deliberately PyTorch-native so later improvements can backpropagate through parts of the pipeline if useful.

Scientifically, this converts learned geometric preferences into an actual fold. The model says which distances are plausible; coordinate optimization asks for a 3D chain whose pairwise distances satisfy those preferences while remaining protein-like.

Computationally, we optimize coordinates directly with automatic differentiation. Mathematically, the potential is an energy `E(x) = -sum_ij log p_ij(bin(||x_i - x_j||))` with a reference correction and chain regularizers. Gradient descent changes coordinates in the direction that lowers this learned energy landscape.

In [7]:
def distogram_energy(coords, bin_edges, pair_log_probs, reference_log_probs):
    distances = torch.cdist(coords, coords).clamp_min(1e-6)
    bin_idx = torch.bucketize(distances, bin_edges[1:-1])
    observed = pair_log_probs.gather(-1, bin_idx[..., None]).squeeze(-1)
    reference = reference_log_probs[bin_idx]
    mask = torch.triu(torch.ones_like(distances, dtype=torch.bool), diagonal=2)
    return -(observed - reference)[mask].mean()

def optimize_trace(pair_logits, steps: int = 300, lr: float = 0.05):
    L, _, bins = pair_logits.shape
    dev = pair_logits.device
    bin_edges = torch.linspace(2.0, 22.0, bins + 1, device=dev)
    pair_log_probs = pair_logits.log_softmax(dim=-1)
    reference = torch.full((bins,), -math.log(bins), device=dev)
    coords = torch.randn(L, 3, device=dev, requires_grad=True)
    opt = torch.optim.Adam([coords], lr=lr)
    history = []
    for step in range(steps):
        opt.zero_grad(set_to_none=True)
        energy = distogram_energy(coords, bin_edges, pair_log_probs, reference)
        ca = (coords[1:] - coords[:-1]).norm(dim=-1)
        chain_loss = ((ca - 3.8) ** 2).mean()
        loss = energy + 0.05 * chain_loss
        loss.backward()
        opt.step()
        if step % 50 == 0 or step == steps - 1:
            history.append({"step": step, "loss": float(loss.detach().cpu()), "energy": float(energy.detach().cpu())})
    return coords.detach(), history

model.eval()
with torch.no_grad():
    logits = model(batch["pair"][None].to(device()))["distogram_logits"][0]
coords, hist = optimize_trace(logits, steps=20)
print(hist[-1], tuple(coords.shape))

{'step': 19, 'loss': 0.7935764193534851, 'energy': 0.786684513092041} (96, 3)


## Step 6 - Scoring and improvement loop

Final scoring should use external structural metrics such as TM-score/US-align and CASP-style GDT. The notebook writes score schemas and experiment registries rather than hard-coding a single binary path.

Scientifically, score choice determines what "reproduction" means. TM-score and GDT measure global fold accuracy, while later additions such as lDDT or violation counts can expose local geometry failures that a global metric may hide.

Computationally, we separate metric schemas from metric binaries because clusters differ in installed tools. Mathematically, each experiment row is a paired comparison between predicted coordinates `X_hat` and reference coordinates `X`, after alignment or distance-based matching. The registry also prevents enhanced experiments from being mistaken for faithful reproduction.

In [8]:
score_schema = {
    "target_id": "T0986s2",
    "prediction_path": "results/senior_2020/structures/T0986s2/model_0.pdb",
    "truth_path": "data/senior_2020/raw/casp13_targets/T0986s2.pdb",
    "tm_score": None,
    "gdt_ts": None,
    "faithful": True,
    "implementation": "own_pytorch",
}
experiments = [
    {"name": "senior_faithful_resnet_distogram", "faithful": True, "change": "paper-era data, own residual distogram model"},
    {"name": "senior_modern_msa", "faithful": False, "change": "new sequence databases"},
    {"name": "senior_af2_prior_potential", "faithful": False, "change": "AF2-like prior in coordinate optimizer"},
]
write_text(paths["metrics"] / "score_schema.json", json.dumps(score_schema, indent=2))
write_text(RUN_DIR / "experiment_registry.json", json.dumps(experiments, indent=2))
print(json.dumps(experiments, indent=2))

[
  {
    "name": "senior_faithful_resnet_distogram",
    "faithful": true,
    "change": "paper-era data, own residual distogram model"
  },
  {
    "name": "senior_modern_msa",
    "faithful": false,
    "change": "new sequence databases"
  },
  {
    "name": "senior_af2_prior_potential",
    "faithful": false,
    "change": "AF2-like prior in coordinate optimizer"
  }
]


## Cluster execution template

The notebooks are deliberately importable and runnable from `papermill`, `jupyter nbconvert --execute`, or ordinary notebook execution. For long runs on the cluster, the code writes a small SLURM script that executes this notebook with parameters rather than keeping GPU time tied to the browser session.

In [9]:
slurm = paths["slurm"] / "train_senior_distogram.sbatch"
slurm.write_text(f"""#!/usr/bin/env bash
#SBATCH --job-name=senior-dist
#SBATCH --partition=gpu
#SBATCH --gres=gpu:1
#SBATCH --cpus-per-task=16
#SBATCH --mem=128G
#SBATCH --time=24:00:00
#SBATCH --output={paths['slurm']}/%x-%j.out
set -euo pipefail
cd "{PROJECT_ROOT}"
jupyter nbconvert --to notebook --execute 01_Senior_2020_AlphaFold_CASP13_Reproduction.ipynb --output results/senior_2020/executed.ipynb
""", encoding="utf-8")
print(slurm.read_text(encoding="utf-8"))

#!/usr/bin/env bash
#SBATCH --job-name=senior-dist
#SBATCH --partition=gpu
#SBATCH --gres=gpu:1
#SBATCH --cpus-per-task=16
#SBATCH --mem=128G
#SBATCH --time=24:00:00
#SBATCH --output=/workspace/runs/senior_2020/slurm/%x-%j.out
set -euo pipefail
cd "/workspace"
jupyter nbconvert --to notebook --execute 01_Senior_2020_AlphaFold_CASP13_Reproduction.ipynb --output results/senior_2020/executed.ipynb



## Step 7 - Example-protein comparison against structures and Senior predictions

This section turns a training run into a scientific comparison on concrete proteins. The next cells load three structures for each target: the experimental reference, our model prediction, and the Senior et al. paper prediction. If you have the paper predictions as PDB files, place them under `data/senior_2020/senior_paper_predictions/` or `results/senior_2020/senior_paper_predictions/` with filenames that contain the target id.

Scientifically, this is the first honest visual and numerical test of whether our reproduction is learning the same geometric signal as Senior et al. A loss curve can improve while structures remain wrong; a structure comparison exposes fold-level failures, long-range-contact errors, and coordinate artifacts.

Computationally, we parse C-alpha traces from PDB files, align predicted traces to the experimental trace with the Kabsch algorithm, and compute simple metrics. Mathematically, Kabsch solves `argmin_R,t ||R X_hat + t - X||_F` over rotations and translations. We then compare aligned coordinates, distance matrices `D_ij`, and contact maps so both coordinate-level and topology-level errors are visible.

In [10]:
import matplotlib.pyplot as plt
import pandas as pd
from urllib.request import urlretrieve

COMPARISON_MANIFEST = DATA_DIR / "comparison_targets.json"
PLOT_DIR = RESULT_DIR / "plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)
COMPARISON_TEMPLATE = DATA_DIR / "comparison_targets.template.json"
PAPER_PREDICTION_DIRS = [
    DATA_DIR / "senior_paper_predictions",
    RESULT_DIR / "senior_paper_predictions",
]
TRUTH_DIRS = [
    DATA_DIR / "raw" / "casp13_targets",
    DATA_DIR / "casp13_targets",
    DATA_DIR / "truth",
]
OUR_PREDICTION_DIRS = [
    RESULT_DIR / "structures",
    RESULT_DIR / "predictions",
]

def candidate_structure_files(target_id: str, roots: list[Path]) -> list[Path]:
    suffixes = ["*.pdb", "*.ent", "*.cif", "*.mmcif"]
    matches = []
    for root in roots:
        if not root.exists():
            continue
        for suffix in suffixes:
            matches.extend(root.rglob(f"*{target_id}*{suffix[1:]}"))
            target_dir = root / target_id
            if target_dir.exists():
                matches.extend(target_dir.glob(suffix))
    return sorted(set(p for p in matches if p.is_file()))

def all_structure_files(roots: list[Path]) -> list[Path]:
    files = []
    for root in roots:
        if root.exists():
            for suffix in ["*.pdb", "*.ent", "*.cif", "*.mmcif"]:
                files.extend(root.rglob(suffix))
    return sorted(set(p for p in files if p.is_file()))

def rough_target_id(path: Path) -> str:
    # Prefer the containing target folder when present; otherwise use the stem.
    if path.parent.name not in {"structures", "predictions", "casp13_targets", "truth", "senior_paper_predictions", "raw"}:
        return path.parent.name
    stem = path.stem
    for token in [".", "_model", "_rank", "_prediction", "_pred", "_native", "_truth"]:
        stem = stem.split(token)[0]
    return stem

def discover_comparison_rows() -> list[dict]:
    truth_files = all_structure_files(TRUTH_DIRS)
    ours_files = all_structure_files(OUR_PREDICTION_DIRS)
    senior_files = all_structure_files(PAPER_PREDICTION_DIRS)
    truth_by_id = {rough_target_id(p): p for p in truth_files}
    ours_by_id = {rough_target_id(p): p for p in ours_files}
    senior_by_id = {rough_target_id(p): p for p in senior_files}
    target_ids = sorted(set(truth_by_id) & set(ours_by_id))
    rows = []
    for target_id in target_ids:
        rows.append({
            "target_id": target_id,
            "truth_pdb": str(truth_by_id[target_id]),
            "ours_pdb": str(ours_by_id[target_id]),
            "senior_paper_pdb": str(senior_by_id.get(target_id, "")),
        })
    return rows

template_rows = [
    {
        "target_id": "replace_with_target_id",
        "truth_pdb_id": "optional_rcsb_pdb_id",
        "truth_url": "optional_direct_download_url",
        "ours_url": "optional_direct_download_url",
        "senior_paper_url": "optional_direct_download_url",
        "truth_pdb": "data/senior_2020/raw/casp13_targets/replace_with_target_id.pdb",
        "ours_pdb": "results/senior_2020/structures/replace_with_target_id/model_0.pdb",
        "senior_paper_pdb": "data/senior_2020/senior_paper_predictions/replace_with_target_id.pdb"
    }
]
write_text(COMPARISON_TEMPLATE, json.dumps(template_rows, indent=2))

if COMPARISON_MANIFEST.exists():
    comparison_rows = json.loads(COMPARISON_MANIFEST.read_text(encoding="utf-8"))
    print(f"Loaded comparison manifest: {COMPARISON_MANIFEST}")
else:
    comparison_rows = discover_comparison_rows()
    if comparison_rows:
        write_text(COMPARISON_MANIFEST, json.dumps(comparison_rows, indent=2))
        print(f"Discovered comparison files and wrote: {COMPARISON_MANIFEST}")
    else:
        print("No complete truth+ours PDB pairs were discovered, so no plots can be made yet.")
        print("Put experimental structures in one of:")
        for p in TRUTH_DIRS:
            print("  truth:", p)
        print("Put our predicted structures in one of:")
        for p in OUR_PREDICTION_DIRS:
            print("  ours :", p)
        print("Optional Senior et al. prediction PDBs go in one of:")
        for p in PAPER_PREDICTION_DIRS:
            print("  senior:", p)
        print(f"Template manifest written to: {COMPARISON_TEMPLATE}")

print("Comparison rows:")
print(json.dumps(comparison_rows, indent=2))

Loaded comparison manifest: /workspace/data/senior_2020/comparison_targets.json
Comparison rows:
[
  {
    "target_id": "T0986s2",
    "truth_pdb_id": "6D7Y",
    "truth_url": "",
    "ours_url": "",
    "senior_paper_url": "https://predictioncenter.org/casp13/MODELS_PDB/T0986s2-D1/T0986s2TS043_1-D1.pdb",
    "truth_pdb": "data/senior_2020/raw/casp13_targets/T0986s2.pdb",
    "ours_pdb": "results/senior_2020/structures/T0986s2/model_0.pdb",
    "senior_paper_pdb": "data/senior_2020/senior_paper_predictions/T0986s2.pdb"
  }
]


In [11]:
def download_if_missing(url: str | None, destination: Path) -> bool:
    if not url or str(url).startswith("optional_"):
        return False
    if destination.exists() and destination.stat().st_size > 0:
        return True
    destination.parent.mkdir(parents=True, exist_ok=True)
    print(f"Downloading {url} -> {destination}")
    urlretrieve(url, destination)
    return destination.exists() and destination.stat().st_size > 0

def download_rcsb_pdb_if_missing(pdb_id: str | None, destination: Path) -> bool:
    if not pdb_id or str(pdb_id).startswith("optional_"):
        return False
    pdb_id = pdb_id.strip().lower()
    return download_if_missing(f"https://files.rcsb.org/download/{pdb_id}.pdb", destination)

def materialize_comparison_inputs(rows: list[dict]) -> list[dict]:
    materialized = []
    for row in rows:
        row = dict(row)
        target_id = row["target_id"]
        truth_path = Path(row.get("truth_pdb") or (DATA_DIR / "raw" / "casp13_targets" / f"{target_id}.pdb"))
        ours_path = Path(row.get("ours_pdb") or (RESULT_DIR / "structures" / target_id / "model_0.pdb"))
        senior_path = Path(row.get("senior_paper_pdb") or (DATA_DIR / "senior_paper_predictions" / f"{target_id}.pdb"))

        if not truth_path.exists():
            download_rcsb_pdb_if_missing(row.get("truth_pdb_id"), truth_path)
        if not truth_path.exists():
            download_if_missing(row.get("truth_url"), truth_path)
        if not ours_path.exists():
            download_if_missing(row.get("ours_url"), ours_path)
        if not senior_path.exists():
            download_if_missing(row.get("senior_paper_url"), senior_path)

        row["truth_pdb"] = str(truth_path)
        row["ours_pdb"] = str(ours_path)
        row["senior_paper_pdb"] = str(senior_path)
        materialized.append(row)

    return materialized

comparison_rows = materialize_comparison_inputs(comparison_rows)
if comparison_rows:
    write_text(COMPARISON_MANIFEST, json.dumps(comparison_rows, indent=2))
    print(f"Updated local comparison manifest: {COMPARISON_MANIFEST}")
else:
    print("No comparison rows to download. Add rows to the manifest template first.")

URLError: <urlopen error Tunnel connection failed: 403 Forbidden>

In [ ]:
def parse_ca_pdb(path: Path):
    rows = []
    for line in path.read_text(errors="ignore").splitlines():
        if not line.startswith(("ATOM  ", "HETATM")):
            continue
        atom = line[12:16].strip()
        if atom != "CA":
            continue
        altloc = line[16].strip()
        if altloc not in {"", "A"}:
            continue
        try:
            chain = line[21].strip() or "_"
            resseq = int(line[22:26])
            icode = line[26].strip()
            coord = np.array([float(line[30:38]), float(line[38:46]), float(line[46:54])], dtype=np.float64)
        except ValueError:
            continue
        rows.append({"key": (chain, resseq, icode), "chain": chain, "resseq": resseq, "coord": coord})
    return rows

def paired_ca_coords(reference_path: Path, prediction_path: Path):
    ref_rows = parse_ca_pdb(reference_path)
    pred_rows = parse_ca_pdb(prediction_path)
    ref_by_key = {r["key"]: r["coord"] for r in ref_rows}
    pred_by_key = {r["key"]: r["coord"] for r in pred_rows}
    common = [r["key"] for r in ref_rows if r["key"] in pred_by_key]
    if len(common) >= 3:
        return np.stack([ref_by_key[k] for k in common]), np.stack([pred_by_key[k] for k in common]), common
    n = min(len(ref_rows), len(pred_rows))
    if n < 3:
        raise ValueError(f"Need at least 3 paired C-alpha atoms: {reference_path}, {prediction_path}")
    return np.stack([r["coord"] for r in ref_rows[:n]]), np.stack([r["coord"] for r in pred_rows[:n]]), [r["key"] for r in ref_rows[:n]]

def kabsch_align(mobile: np.ndarray, target: np.ndarray):
    mobile_center = mobile.mean(axis=0)
    target_center = target.mean(axis=0)
    X = mobile - mobile_center
    Y = target - target_center
    C = X.T @ Y
    V, _, Wt = np.linalg.svd(C)
    d = np.sign(np.linalg.det(V @ Wt))
    R = V @ np.diag([1.0, 1.0, d]) @ Wt
    return X @ R + target_center

def distance_matrix(coords: np.ndarray):
    diff = coords[:, None, :] - coords[None, :, :]
    return np.sqrt((diff * diff).sum(axis=-1))

def structure_metrics(reference: np.ndarray, prediction: np.ndarray):
    aligned = kabsch_align(prediction, reference)
    per_residue_error = np.linalg.norm(aligned - reference, axis=-1)
    rmsd = float(np.sqrt(np.mean(per_residue_error ** 2)))
    L = len(reference)
    d0 = max(0.5, 1.24 * max(L - 15, 1) ** (1 / 3) - 1.8)
    tm_like = float(np.mean(1.0 / (1.0 + (per_residue_error / d0) ** 2)))
    gdt_ts_like = float(np.mean([np.mean(per_residue_error <= t) for t in [1.0, 2.0, 4.0, 8.0]]))
    ref_d = distance_matrix(reference)
    pred_d = distance_matrix(aligned)
    sep_mask = np.triu(np.ones((L, L), dtype=bool), k=6)
    dist_mae = float(np.mean(np.abs(ref_d[sep_mask] - pred_d[sep_mask]))) if sep_mask.any() else np.nan
    ref_contacts = (ref_d < 8.0) & sep_mask
    pred_contacts = (pred_d < 8.0) & sep_mask
    contact_precision = float((ref_contacts & pred_contacts).sum() / max(pred_contacts.sum(), 1))
    contact_recall = float((ref_contacts & pred_contacts).sum() / max(ref_contacts.sum(), 1))
    return {
        "aligned": aligned,
        "per_residue_error": per_residue_error,
        "rmsd_ca": rmsd,
        "tm_like": tm_like,
        "gdt_ts_like": gdt_ts_like,
        "distance_mae_long_range": dist_mae,
        "contact_precision_8A": contact_precision,
        "contact_recall_8A": contact_recall,
        "reference_distance": ref_d,
        "prediction_distance": pred_d,
    }

def resolve_comparison_paths(row: dict):
    target_id = row["target_id"]
    truth = Path(row.get("truth_pdb", ""))
    ours = Path(row.get("ours_pdb", ""))
    senior = Path(row.get("senior_paper_pdb", ""))
    if not truth.exists():
        candidates = candidate_structure_files(target_id, TRUTH_DIRS)
        truth = candidates[0] if candidates else truth
    if not ours.exists():
        candidates = candidate_structure_files(target_id, OUR_PREDICTION_DIRS)
        ours = candidates[0] if candidates else ours
    if not senior.exists():
        candidates = candidate_structure_files(target_id, PAPER_PREDICTION_DIRS)
        senior = candidates[0] if candidates else senior
    return truth, ours, senior

In [ ]:
comparison_records = []
comparison_payloads = {}

for row in comparison_rows:
    target_id = row["target_id"]
    truth_path, ours_path, senior_path = resolve_comparison_paths(row)
    print(f"\nTarget {target_id}")
    print("  truth :", truth_path, truth_path.exists())
    print("  ours  :", ours_path, ours_path.exists())
    print("  Senior:", senior_path, senior_path.exists())
    if not truth_path.exists() or not ours_path.exists():
        print("  Skipping: need at least truth_pdb and ours_pdb.")
        continue

    reference, ours_coords, _ = paired_ca_coords(truth_path, ours_path)
    ours_metrics = structure_metrics(reference, ours_coords)
    comparison_records.append({
        "target_id": target_id,
        "method": "ours",
        "n_residues": len(reference),
        **{k: v for k, v in ours_metrics.items() if isinstance(v, float)},
    })
    comparison_payloads[(target_id, "ours")] = {"reference": reference, "prediction": ours_metrics["aligned"], **ours_metrics}

    if senior_path.exists():
        reference_s, senior_coords, _ = paired_ca_coords(truth_path, senior_path)
        senior_metrics = structure_metrics(reference_s, senior_coords)
        comparison_records.append({
            "target_id": target_id,
            "method": "Senior et al.",
            "n_residues": len(reference_s),
            **{k: v for k, v in senior_metrics.items() if isinstance(v, float)},
        })
        comparison_payloads[(target_id, "Senior et al.")] = {"reference": reference_s, "prediction": senior_metrics["aligned"], **senior_metrics}

metrics_df = pd.DataFrame(comparison_records)
metrics_path = paths["metrics"] / "example_structure_comparison.csv"
if not metrics_df.empty:
    metrics_df.to_csv(metrics_path, index=False)
    display(metrics_df.sort_values(["target_id", "method"]))
    print(f"Saved metrics to {metrics_path}")
else:
    print("No complete comparisons found yet.")
    print(f"Edit {COMPARISON_MANIFEST} using {COMPARISON_TEMPLATE} as a template, or place matching PDB files in the documented folders and rerun this section.")
    print("A complete row needs at least target_id, truth_pdb, and ours_pdb. senior_paper_pdb is optional.")

In [ ]:
def plot_ca_trace(ax, coords: np.ndarray, title: str, color: str):
    ax.plot(coords[:, 0], coords[:, 1], coords[:, 2], color=color, linewidth=1.6)
    ax.scatter(coords[0, 0], coords[0, 1], coords[0, 2], color=color, s=30, marker="o")
    ax.scatter(coords[-1, 0], coords[-1, 1], coords[-1, 2], color=color, s=30, marker="x")
    ax.set_title(title)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_zlabel("z")

def plot_target_comparison(target_id: str):
    methods = [m for (tid, m) in comparison_payloads if tid == target_id]
    if not methods:
        print(f"No payloads for {target_id}")
        return

    fig = plt.figure(figsize=(5 * (len(methods) + 1), 4.5))
    reference = comparison_payloads[(target_id, methods[0])]["reference"]
    ax = fig.add_subplot(1, len(methods) + 1, 1, projection="3d")
    plot_ca_trace(ax, reference, f"{target_id} experimental", "black")
    for idx, method in enumerate(methods, start=2):
        payload = comparison_payloads[(target_id, method)]
        ax = fig.add_subplot(1, len(methods) + 1, idx, projection="3d")
        plot_ca_trace(ax, payload["reference"], "reference", "lightgray")
        plot_ca_trace(ax, payload["prediction"], method, "tab:blue" if method == "ours" else "tab:orange")
    fig.tight_layout()
    fig.savefig(PLOT_DIR / f"{target_id}_ca_trace_comparison.png", dpi=200, bbox_inches="tight")
    plt.show()

    fig, axes = plt.subplots(len(methods), 3, figsize=(13, 4 * len(methods)), squeeze=False)
    for row_idx, method in enumerate(methods):
        payload = comparison_payloads[(target_id, method)]
        ref_d = payload["reference_distance"]
        pred_d = payload["prediction_distance"]
        err_d = np.abs(ref_d - pred_d)
        for ax, mat, title in [
            (axes[row_idx, 0], ref_d, "experimental distance map"),
            (axes[row_idx, 1], pred_d, f"{method} distance map"),
            (axes[row_idx, 2], err_d, f"{method} |distance error|"),
        ]:
            im = ax.imshow(mat, cmap="viridis" if "error" not in title else "magma")
            ax.set_title(f"{target_id}: {title}")
            ax.set_xlabel("residue")
            ax.set_ylabel("residue")
            fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(PLOT_DIR / f"{target_id}_distance_maps.png", dpi=200, bbox_inches="tight")
    plt.show()

    fig, ax = plt.subplots(figsize=(10, 4))
    for method in methods:
        payload = comparison_payloads[(target_id, method)]
        ax.plot(payload["per_residue_error"], label=method)
    ax.set_title(f"{target_id}: per-residue C-alpha error after Kabsch alignment")
    ax.set_xlabel("paired residue index")
    ax.set_ylabel("error (Angstrom)")
    ax.legend()
    ax.grid(alpha=0.25)
    fig.savefig(PLOT_DIR / f"{target_id}_per_residue_error.png", dpi=200, bbox_inches="tight")
    plt.show()

if not metrics_df.empty:
    for target_id in metrics_df["target_id"].unique():
        plot_target_comparison(target_id)

In [ ]:
if not metrics_df.empty:
    score_cols = ["rmsd_ca", "tm_like", "gdt_ts_like", "distance_mae_long_range", "contact_precision_8A", "contact_recall_8A"]
    available_score_cols = [c for c in score_cols if c in metrics_df.columns]
    for metric in available_score_cols:
        pivot = metrics_df.pivot(index="target_id", columns="method", values=metric)
        ax = pivot.plot(kind="bar", figsize=(10, 4), rot=45)
        ax.set_title(f"Our Senior reproduction vs Senior et al.: {metric}")
        ax.set_ylabel(metric)
        ax.grid(axis="y", alpha=0.25)
        plt.tight_layout()
        plt.savefig(PLOT_DIR / f"method_comparison_{metric}.png", dpi=200, bbox_inches="tight")
        plt.show()

    if {"ours", "Senior et al."}.issubset(set(metrics_df["method"])):
        paired = metrics_df.pivot(index="target_id", columns="method", values="tm_like").dropna()
        if not paired.empty:
            paired["ours_minus_senior"] = paired["ours"] - paired["Senior et al."]
            colors = np.where(paired["ours_minus_senior"] >= 0, "tab:green", "tab:red")
            ax = paired["ours_minus_senior"].plot(kind="bar", figsize=(10, 3), color=colors)
            ax.axhline(0, color="black", linewidth=1)
            ax.set_title("TM-like delta: ours minus Senior et al.")
            ax.set_ylabel("delta")
            ax.grid(axis="y", alpha=0.25)
            plt.tight_layout()
            plt.savefig(PLOT_DIR / "ours_minus_senior_tm_like_delta.png", dpi=200, bbox_inches="tight")
            plt.show()
            display(paired)